# DATA 266 Lab 1 — Part 1: GPT-Style LLM From Scratch

This notebook implements a character-level GPT-style language model **without** using
`nn.Transformer`, `nn.TransformerEncoder`, or `nn.MultiheadAttention`.

It is designed to produce the submission artifacts automatically:

```text
Part1/
├── notebook.ipynb                <- save this notebook here after execution
├── model.pt
├── metrics.csv
├── training_log.txt
├── failure_analysis.md
├── config.json
└── outputs/
    ├── loss_curve.png
    ├── generated_samples.txt
    └── generated_samples.json
```

### Before the final submission
1. Run every cell on the real dataset.
2. Keep all outputs visible in the notebook.
3. Confirm that `EPOCHS >= 10`.
4. Review the three generated failure cases yourself and edit `failure_analysis.md` if needed.
5. Copy the executed notebook into the `Part1` folder.

In [ ]:
# Install packages into the currently selected VS Code/Jupyter kernel.
%pip -q install datasets pandas matplotlib


In [ ]:
from pathlib import Path
import os, sys, json, math, time, random, logging, platform
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset

SEED = 266
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Configuration

This version is prepared for **VS Code + RunPod**.

- Open the repository folder in the VS Code window connected to RunPod.
- The notebook automatically finds the repository root.
- No Google Drive mount and no `/content/...` paths are required.
- You may optionally set `DATA266_REPO_ROOT`, `DATA266_MEMBER_NAME`, or `DATA266_RUN_MODE` as environment variables.


In [ ]:
# ===== VS CODE / RUNPOD REPOSITORY SETUP =====
from pathlib import Path
import os

# Optional override. Usually you do not need to set this.
repo_env = os.environ.get("DATA266_REPO_ROOT", "").strip()

if repo_env:
    REPO_ROOT = Path(repo_env).expanduser().resolve()
else:
    # VS Code notebooks normally start from the opened workspace folder.
    # Search the current folder and its parents for this repository's markers.
    start = Path.cwd().resolve()
    REPO_ROOT = None

    for candidate in [start, *start.parents]:
        if (candidate / "task1_llm").exists() and (candidate / "reproducibility").exists():
            REPO_ROOT = candidate
            break

    if REPO_ROOT is None:
        raise RuntimeError(
            "Could not find the repository root. "
            "In the RunPod VS Code window, use File > Open Folder and open the team repository folder, "
            "then restart the kernel and run the notebook again."
        )

print("Running from:", Path.cwd())

MEMBER_NAME = os.environ.get("DATA266_MEMBER_NAME", "Kavya").strip()
RUN_MODE = os.environ.get("DATA266_RUN_MODE", "full").strip().lower()
if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("DATA266_RUN_MODE must be 'smoke' or 'full'.")


print("Repository root:", REPO_ROOT)
print("Member name:", MEMBER_NAME)
print("Run mode:", RUN_MODE)


TASK_ROOT = REPO_ROOT / "task1_llm"
MEMBER_DIR = TASK_ROOT / MEMBER_NAME
PART_DIR = MEMBER_DIR
SRC_DIR = MEMBER_DIR / "src"
DATA_PROCESSED_DIR = MEMBER_DIR / "data_processed"
CKPT_DIR = MEMBER_DIR / "checkpoints"
OUTPUT_DIR = MEMBER_DIR / "outputs"
MANIFEST_DIR = REPO_ROOT / "reproducibility" / "manifests"
RAW_LOG_DIR = REPO_ROOT / "reproducibility" / "raw_logs"

RUN_CKPT_DIR = CKPT_DIR / RUN_MODE
RUN_OUTPUT_DIR = OUTPUT_DIR if RUN_MODE == "full" else OUTPUT_DIR / "smoke"
ARTIFACT_BASE = MEMBER_DIR if RUN_MODE == "full" else RUN_OUTPUT_DIR
for d in [MEMBER_DIR, SRC_DIR, DATA_PROCESSED_DIR, CKPT_DIR, RUN_CKPT_DIR,
          OUTPUT_DIR, RUN_OUTPUT_DIR, MANIFEST_DIR, RAW_LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Full run follows the assignment's 100K / 10K story split.
TRAIN_STORIES = 100_000 if RUN_MODE == "full" else 2_000
VAL_STORIES = 10_000 if RUN_MODE == "full" else 200
BLOCK_SIZE = 128

D_MODEL = 256
N_HEADS = 8
N_LAYERS = 4
DROPOUT = 0.10
BATCH_SIZE = 64
EPOCHS = 10 if RUN_MODE == "full" else 1
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 500 if RUN_MODE == "full" else 10

# Computed from the actual token count after tokenization for the full run.
STEPS_PER_EPOCH = None
VAL_STEPS = None

CONFIG = {
    "run_mode": RUN_MODE,
    "seed": SEED,
    "train_stories": TRAIN_STORIES,
    "val_stories": VAL_STORIES,
    "block_size": BLOCK_SIZE,
    "d_model": D_MODEL,
    "n_heads": N_HEADS,
    "n_layers": N_LAYERS,
    "dropout": DROPOUT,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_steps": WARMUP_STEPS,
    "device": str(DEVICE),
}

CONFIG_PATH = (MEMBER_DIR / "config.json") if RUN_MODE == "full" else (RUN_OUTPUT_DIR / "config_smoke.json")

def assert_config_compatible(saved_config, current_config, label="checkpoint"):
    """Refuse to resume when training-relevant settings differ."""
    ignored = {"device"}
    keys = sorted((set(saved_config) | set(current_config)) - ignored)
    mismatches = []
    for key in keys:
        if saved_config.get(key) != current_config.get(key):
            mismatches.append(
                f"{key}: saved={saved_config.get(key)!r}, current={current_config.get(key)!r}"
            )
    if mismatches:
        details = "\n  - " + "\n  - ".join(mismatches)
        raise RuntimeError(
            f"Refusing to reuse {label}: its configuration does not match this run.{details}\n"
            "Use the correct RUN_MODE/config, or intentionally remove that run's checkpoint directory."
        )

In [ ]:
# Keep smoke and full raw logs separate so the evidence trail is never mixed.
LOG_PATH = RAW_LOG_DIR / f"{MEMBER_NAME}_task1_{RUN_MODE}.log"

logger = logging.getLogger(f"part1_{RUN_MODE}")
logger.setLevel(logging.INFO)
logger.handlers.clear()

# Append supports a legitimate resumed run without editing old log lines.
file_handler = logging.FileHandler(LOG_PATH, mode="a")
stream_handler = logging.StreamHandler(sys.stdout)
formatter = logging.Formatter("%(asctime)s | %(message)s")
file_handler.setFormatter(formatter)
stream_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.addHandler(stream_handler)

logger.info("Part 1 run/session started")
logger.info(f"Run mode: {RUN_MODE}")
logger.info(f"Python: {platform.python_version()}")
logger.info(f"PyTorch: {torch.__version__}")
logger.info(f"Device: {DEVICE}")
if torch.cuda.is_available():
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load TinyStories and create an independent 100K/10K split

The split is produced from the TinyStories training set using this notebook's seed.
The text is tokenized at the **character level**.

In [ ]:
raw = load_dataset("roneneldan/TinyStories", split="train")

required = TRAIN_STORIES + VAL_STORIES
if len(raw) < required:
    raise ValueError(f"Dataset has only {len(raw)} rows, but {required} are required.")

# Independent member split.
indices = np.random.default_rng(SEED).permutation(len(raw))[:required]
train_idx = indices[:TRAIN_STORIES]
val_idx = indices[TRAIN_STORIES:]

train_stories = [raw[int(i)]["text"] for i in train_idx]
val_stories = [raw[int(i)]["text"] for i in val_idx]

print("Train stories:", len(train_stories))
print("Validation stories:", len(val_stories))
print("Example:", train_stories[0][:300])

In [ ]:
# Build vocabulary from training text only.
train_text = "\n".join(train_stories)
val_text = "\n".join(val_stories)

chars = sorted(set(train_text))
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}
VOCAB_SIZE = len(chars)
SPACE_ID = char_to_idx.get(" ", 0)

def encode(text):
    return torch.tensor([char_to_idx.get(ch, SPACE_ID) for ch in text], dtype=torch.long)

def decode(ids):
    return "".join(idx_to_char[int(i)] for i in ids)

train_ids = encode(train_text)
val_ids = encode(val_text)

# A full epoch uses a token budget approximately equal to one pass through the corpus.
# Batches are random fixed-length blocks sampled with replacement, so this is a
# token-budget-equivalent epoch rather than a deterministic sequential pass.
if RUN_MODE == "full":
    STEPS_PER_EPOCH = max(1, math.ceil((len(train_ids) - 1) / (BATCH_SIZE * BLOCK_SIZE)))
    VAL_STEPS = max(1, math.ceil((len(val_ids) - 1) / (BATCH_SIZE * BLOCK_SIZE)))
else:
    STEPS_PER_EPOCH = 20
    VAL_STEPS = 10

CONFIG["steps_per_epoch"] = STEPS_PER_EPOCH
CONFIG["val_steps"] = VAL_STEPS
CONFIG["train_character_tokens"] = int(len(train_ids))
CONFIG["validation_character_tokens"] = int(len(val_ids))
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")

print("Vocabulary size:", VOCAB_SIZE)
print("Training characters:", len(train_ids))
print("Validation characters:", len(val_ids))
print("Steps/epoch:", STEPS_PER_EPOCH)
print("Validation steps:", VAL_STEPS)
print("Approx training tokens/epoch:", BATCH_SIZE * BLOCK_SIZE * STEPS_PER_EPOCH)
print("Round-trip check:", decode(train_ids[:100]))

## 3. Fixed-length autoregressive sequences
Each sample has:
- input: characters `t ... t+BLOCK_SIZE-1`
- target: the same sequence shifted by one character

In [ ]:
def get_batch(token_ids, batch_size, block_size, device):
    """Random fixed-length autoregressive blocks without exhaustive sliding windows."""
    max_start = len(token_ids) - block_size - 1
    if max_start <= 0:
        raise ValueError("Token sequence is shorter than BLOCK_SIZE.")

    starts = torch.randint(0, max_start, (batch_size,), dtype=torch.long)
    offsets = torch.arange(block_size, dtype=torch.long).unsqueeze(0)
    positions = starts.unsqueeze(1) + offsets
    x = token_ids[positions]
    y = token_ids[positions + 1]
    return x.to(device, non_blocking=True), y.to(device, non_blocking=True)

xb, yb = get_batch(train_ids, BATCH_SIZE, BLOCK_SIZE, DEVICE)
print("Input batch:", xb.shape, "Target batch:", yb.shape)

## 4. GPT implementation from scratch

The attention computation below is explicitly implemented with:
1. learned Q/K/V projections,
2. scaled dot-product scores,
3. causal masking,
4. softmax,
5. weighted value aggregation.

No prebuilt Transformer or attention module is used.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, block_size, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.scale = self.head_dim ** -0.5

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(block_size, block_size, dtype=torch.bool))
        self.register_buffer("causal_mask", mask.view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.causal_mask[:, :, :T, :T], float("-inf"))
        weights = F.softmax(scores, dim=-1)
        weights = self.attn_dropout(weights)

        out = weights @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


class FeedForward(nn.Module):
    def __init__(self, d_model, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class CharGPT(nn.Module):
    def __init__(self, vocab_size, block_size, d_model, n_heads, n_layers, dropout):
        super().__init__()
        self.block_size = block_size
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, block_size, dropout)
            for _ in range(n_layers)
        ])
        self.final_ln = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        if T > self.block_size:
            raise ValueError("Sequence length exceeds block size")

        positions = torch.arange(T, device=idx.device)
        x = self.token_emb(idx) + self.pos_emb(positions)[None, :, :]
        x = self.dropout(x)

        for block in self.blocks:
            x = block(x)

        x = self.final_ln(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1)
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=300, temperature=0.8, greedy=False):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]

            if greedy:
                next_id = torch.argmax(logits, dim=-1, keepdim=True)
            else:
                logits = logits / max(temperature, 1e-6)
                probs = F.softmax(logits, dim=-1)
                next_id = torch.multinomial(probs, num_samples=1)

            idx = torch.cat([idx, next_id], dim=1)
        return idx

model = CharGPT(
    VOCAB_SIZE, BLOCK_SIZE, D_MODEL, N_HEADS, N_LAYERS, DROPOUT
).to(DEVICE)

PARAM_COUNT = sum(p.numel() for p in model.parameters())
print(model)
print(f"Parameter count: {PARAM_COUNT:,}")

## 5. Training with warm-up + cosine learning-rate schedule

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = EPOCHS * STEPS_PER_EPOCH

def lr_lambda(step):
    if step < WARMUP_STEPS:
        return max(1, step) / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
    progress = min(max(progress, 0.0), 1.0)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
history = []
global_step = 0
nan_count = 0
peak_grad_norm = 0.0
start_epoch = 0
cumulative_training_seconds = 0.0
prior_peak_gpu_memory_mb = 0.0
LATEST_CKPT = RUN_CKPT_DIR / "latest_checkpoint.pt"

if LATEST_CKPT.exists():
    ckpt = torch.load(LATEST_CKPT, map_location=DEVICE)
    assert_config_compatible(ckpt.get("config", {}), CONFIG, label=str(LATEST_CKPT))
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    history = ckpt.get("history", [])
    global_step = ckpt.get("global_step", 0)
    nan_count = ckpt.get("nan_count", 0)
    peak_grad_norm = ckpt.get("peak_grad_norm", 0.0)
    cumulative_training_seconds = ckpt.get("cumulative_training_seconds", 0.0)
    prior_peak_gpu_memory_mb = ckpt.get("peak_gpu_memory_mb", 0.0)
    start_epoch = int(ckpt["epoch"]) + 1
    logger.info(f"Resuming compatible {RUN_MODE} checkpoint from epoch {start_epoch + 1}/{EPOCHS}")

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

session_start = time.perf_counter()

for epoch in range(start_epoch, EPOCHS):
    model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_tokens = 0
    epoch_start = time.perf_counter()

    for step in range(STEPS_PER_EPOCH):
        x, y = get_batch(train_ids, BATCH_SIZE, BLOCK_SIZE, DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits, loss = model(x, y)

        if not torch.isfinite(loss):
            nan_count += 1
            logger.info(f"Non-finite loss at epoch={epoch+1}, step={step}")
            continue

        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        peak_grad_norm = max(peak_grad_norm, float(grad_norm))
        optimizer.step()
        scheduler.step()
        global_step += 1

        with torch.no_grad():
            preds = logits.argmax(dim=-1)
            train_correct += (preds == y).sum().item()
            train_tokens += y.numel()
        train_loss_sum += loss.item() * y.numel()

    train_time = time.perf_counter() - epoch_start
    train_loss = train_loss_sum / max(1, train_tokens)
    train_acc = train_correct / max(1, train_tokens)
    train_tok_s = train_tokens / max(train_time, 1e-9)

    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_tokens = 0
    with torch.no_grad():
        for _ in range(VAL_STEPS):
            x, y = get_batch(val_ids, BATCH_SIZE, BLOCK_SIZE, DEVICE)
            logits, loss = model(x, y)
            preds = logits.argmax(dim=-1)
            val_loss_sum += loss.item() * y.numel()
            val_correct += (preds == y).sum().item()
            val_tokens += y.numel()

    val_loss = val_loss_sum / max(1, val_tokens)
    val_acc = val_correct / max(1, val_tokens)
    row = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_top1_accuracy": train_acc,
        "val_top1_accuracy": val_acc,
        "train_tokens_per_sec": train_tok_s,
        "lr": scheduler.get_last_lr()[0],
    }
    history.append(row)

    logger.info(
        f"Epoch {epoch+1:02d}/{EPOCHS} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} "
        f"| train_acc={train_acc:.4f} | val_acc={val_acc:.4f} | tok/s={train_tok_s:,.0f}"
    )

    session_elapsed = time.perf_counter() - session_start
    current_peak_mb = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0.0
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "history": history,
        "global_step": global_step,
        "nan_count": nan_count,
        "peak_grad_norm": peak_grad_norm,
        "cumulative_training_seconds": cumulative_training_seconds + session_elapsed,
        "peak_gpu_memory_mb": max(prior_peak_gpu_memory_mb, current_peak_mb),
        "config": CONFIG,
        "vocab": chars,
        "char_to_idx": char_to_idx,
        "idx_to_char": idx_to_char,
    }
    torch.save(checkpoint, RUN_CKPT_DIR / f"gpt_epoch_{epoch+1:02d}.pt")
    torch.save(checkpoint, LATEST_CKPT)

session_elapsed = time.perf_counter() - session_start
total_train_time = cumulative_training_seconds + session_elapsed
history_df = pd.DataFrame(history)
if history_df.empty:
    raise RuntimeError("No training history is available. Check the checkpoint/run settings.")
history_df

## 6. Save checkpoint and training curves

In [ ]:
# Compact model checkpoint used for submission (full) or smoke verification (smoke).
final_checkpoint = {
    "model_state_dict": model.state_dict(),
    "config": CONFIG,
    "vocab": chars,
    "char_to_idx": char_to_idx,
    "idx_to_char": idx_to_char,
    "history": history,
}
torch.save(final_checkpoint, RUN_CKPT_DIR / "model.pt")
print("Saved:", RUN_CKPT_DIR / "model.pt")

plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["train_loss"], label="Train")
plt.plot(history_df["epoch"], history_df["val_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.title(f"Part 1 Loss ({RUN_MODE})")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(RUN_OUTPUT_DIR / "loss_curve.png", dpi=160)
plt.show()

## 7. Generation metrics

In [ ]:
def distinct_n(text, n):
    tokens = list(text)
    if len(tokens) < n:
        return 0.0
    ngrams = [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
    return len(set(ngrams)) / max(1, len(ngrams))

def repeated_ngram_rate(text, n=4):
    tokens = list(text)
    if len(tokens) < n:
        return 0.0
    ngrams = [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
    counts = Counter(ngrams)
    repeated_occurrences = sum(c for c in counts.values() if c > 1)
    return repeated_occurrences / max(1, len(ngrams))

prompts = ["Once upon a time", "The little girl", "One day"]
samples = []

gen_start = time.perf_counter()
generated_token_count = 0

for prompt in prompts:
    prompt_ids = encode(prompt).unsqueeze(0).to(DEVICE)
    out = model.generate(prompt_ids, max_new_tokens=400, temperature=0.8, greedy=False)
    generated_token_count += out.shape[1] - prompt_ids.shape[1]
    text = decode(out[0].cpu().tolist())
    samples.append({"prompt": prompt, "text": text})

gen_time = time.perf_counter() - gen_start
generation_tokens_per_sec = generated_token_count / max(gen_time, 1e-9)

with open(RUN_OUTPUT_DIR / "generated_samples.txt", "w", encoding="utf-8") as f:
    for i, s in enumerate(samples, 1):
        f.write(f"===== SAMPLE {i} =====\n{s['text']}\n\n")

(RUN_OUTPUT_DIR / "generated_samples.json").write_text(
    json.dumps(samples, indent=2, ensure_ascii=False), encoding="utf-8"
)

for i, s in enumerate(samples, 1):
    print(f"\n===== SAMPLE {i} =====")
    print(s["text"][:1200])

## 8. Final metrics table

In [ ]:
final_train_loss = float(history_df.iloc[-1]["train_loss"])
final_val_loss = float(history_df.iloc[-1]["val_loss"])
all_generated_text = "\n".join(s["text"] for s in samples)

current_peak_mb = torch.cuda.max_memory_allocated() / (1024**2) if torch.cuda.is_available() else 0.0
metrics = {
    "training_cross_entropy_loss": final_train_loss,
    "validation_cross_entropy_loss": final_val_loss,
    "perplexity": float(math.exp(min(final_val_loss, 20))),
    "bits_per_character": float(final_val_loss / math.log(2)),
    "generalization_gap": float(final_val_loss - final_train_loss),
    "top1_next_character_accuracy": float(history_df.iloc[-1]["val_top1_accuracy"]),
    "distinct_1": distinct_n(all_generated_text, 1),
    "distinct_2": distinct_n(all_generated_text, 2),
    "distinct_3": distinct_n(all_generated_text, 3),
    "repeated_4gram_rate": repeated_ngram_rate(all_generated_text, 4),
    "peak_gradient_norm": peak_grad_norm,
    "nan_count": nan_count,
    "parameter_count": PARAM_COUNT,
    "training_tokens_per_sec": float(history_df["train_tokens_per_sec"].mean()),
    "generation_tokens_per_sec": generation_tokens_per_sec,
    "peak_gpu_memory_mb": max(prior_peak_gpu_memory_mb, current_peak_mb),
    "total_training_time_seconds": total_train_time,
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "steps_per_epoch": STEPS_PER_EPOCH,
    "training_character_tokens": len(train_ids),
    "run_mode": RUN_MODE,
}
metrics_df = pd.DataFrame([metrics])
metrics_path = ARTIFACT_BASE / "metrics_report.csv"
metrics_df.to_csv(metrics_path, index=False)
print("Saved:", metrics_path)
metrics_df.T

## 9. Three generated-text failure cases

The notebook creates a starter file using three generated samples. **Read the actual text and replace
the failure labels/observations with your own analysis before submission.** The viva requires you to
explain these failures yourself.

In [ ]:
# ARTIFACT_BASE was configured above for this member/run mode.
ARTIFACT_BASE.mkdir(parents=True, exist_ok=True)

failure_template = """# Part 1 Failure Analysis

## Failure Case 1
**Generated snippet:**
Once upon a time, there was a little girl named Lily. She loved to play with her toys all day long. One day, her mommy asked her to increase the store to tell her names the little girl what she was doing. Mommy said yes, and Lily put her pink dress on her head. But then, she tripped on a rock and fell down with her pink dress. It went higher and higher until she reached the top. She felt very black and happy.

**Failure type:** Loss of coherence / broken grammar

**Observation:** The story starts clearly, but the sentence "increase the store to tell her names" is grammatically incorrect and does not make sense. The sequence of events also becomes inconsistent when Lily puts a dress on her head and then somehow goes higher and higher.

---

## Failure Case 2
**Generated snippet:**
The little girl loved the caterpillar so much that she decided to swim to get it but the caterpillar was too shy and her beak was in her bed.

The caterpillar was very sad and he didn't know what to do. He tried to put it back in his bed, but it was too late. Lily's mom came into the kitchen and saw the mess. She saw the mess and the cookies. She said, "Why is the cookie easy? I want to drink some water."

**Failure type:** Loss of coherence / repetition / broken grammar

**Observation:** The story mixes unrelated ideas. The caterpillar suddenly has a beak and is placed in a bed, which does not match the earlier context. "She saw the mess" is repeated, and "Why is the cookie easy?" does not make logical sense.

---

## Failure Case 3
**Generated snippet:**
One day, he decided to go for a walk outside and see all the beautiful flowers she could find. It was a kind woman and a queen who was very happy. She said, "Thank you, those are so good for my butterfly. You were proud of you."

The two of them thought for a moment, then said, "You are welcome, little butterfly. You are a good friend."

The little butterfly drove away and Lily was happy.

**Failure type:** Loss of coherence / hallucination

**Observation:** The model changes pronouns and characters inconsistently. New characters such as a kind woman, queen, and butterfly appear without explanation. "You were proud of you" is grammatically incorrect, and the butterfly driving away is unrealistic in the context.
"""

output_file = ARTIFACT_BASE / "failure_analysis.md"

output_file.write_text(
    failure_template,
    encoding="utf-8"
)

print("Saved:", output_file)
print(failure_template)

## 10. Final artifact check

In [ ]:
for p in sorted(PART_DIR.rglob("*")):
    if p.is_file():
        print(p.relative_to(PART_DIR))

In [ ]:

# Create results.md from the actual run.
results_md = f"""# Part 1 Results

## Architecture
- Character-level GPT-style language model built from scratch
- Transformer blocks: {N_LAYERS}
- Hidden size: {D_MODEL}
- Attention heads: {N_HEADS}
- Context length: {BLOCK_SIZE}
- Dropout: {DROPOUT}
- Parameters: {PARAM_COUNT:,}

## Training
- Epochs: {EPOCHS}
- Batch size: {BATCH_SIZE}
- Steps per epoch (derived from corpus token count for full run): {STEPS_PER_EPOCH}
- Epoch definition: random blocks with a token budget approximately equal to one corpus pass
- Validation steps per epoch: {VAL_STEPS}
- Optimizer: AdamW
- Initial learning rate: {LEARNING_RATE}
- Warm-up steps: {WARMUP_STEPS}
- Device: {metrics['device']}
- Total training time (s): {metrics['total_training_time_seconds']:.2f}

## Final Metrics
- Training loss: {metrics['training_cross_entropy_loss']:.6f}
- Validation loss: {metrics['validation_cross_entropy_loss']:.6f}
- Perplexity: {metrics['perplexity']:.6f}
- Bits/character: {metrics['bits_per_character']:.6f}
- Generalization gap: {metrics['generalization_gap']:.6f}
- Top-1 next-character accuracy: {metrics['top1_next_character_accuracy']:.6f}
- Distinct-1: {metrics['distinct_1']:.6f}
- Distinct-2: {metrics['distinct_2']:.6f}
- Distinct-3: {metrics['distinct_3']:.6f}
- Repeated 4-gram rate: {metrics['repeated_4gram_rate']:.6f}
- Peak gradient norm: {metrics['peak_gradient_norm']:.6f}
- NaN count: {metrics['nan_count']}
- Training tokens/sec: {metrics['training_tokens_per_sec']:.2f}
- Generation tokens/sec: {metrics['generation_tokens_per_sec']:.2f}
- Peak GPU memory (MB): {metrics['peak_gpu_memory_mb']:.2f}

## Files
- Checkpoint: `checkpoints/full/model.pt`
- Metrics: `metrics_report.csv`
- Failure analysis: `failure_analysis.md`
- Outputs: `outputs/`
"""
(ARTIFACT_BASE / "results.md").write_text(results_md, encoding="utf-8")
print(results_md)


In [ ]:

# Save environment/package manifest for reproducibility.
import subprocess
manifest_path = MANIFEST_DIR / f"{MEMBER_NAME}_task1_manifest.txt"

with open(manifest_path, "w", encoding="utf-8") as f:
    f.write(f"Python: {platform.python_version()}\n")
    f.write(f"PyTorch: {torch.__version__}\n")
    f.write(f"Device: {DEVICE}\n")
    if torch.cuda.is_available():
        f.write(f"GPU: {torch.cuda.get_device_name(0)}\n")
    f.write("\n--- pip freeze ---\n")
    try:
        freeze = subprocess.check_output(
            [sys.executable, "-m", "pip", "freeze"], text=True
        )
        f.write(freeze)
    except Exception as e:
        f.write(f"Could not capture pip freeze: {e}\n")

print("Saved manifest:", manifest_path)
